
- We use system_prompt_advanced
- The model is an ollama model llama3:8b
- We let the model  to reason on the fields he needs to fill in based on the input
- If feedback is that not all required fields are determined, the agent will ask to the point question to receive the extra information of the user. All previous content of that session will be given to the model (langchain) to the model to generate the best output
- The output proposal will be shown to the user, he can confirm with "c" or not confirm with "n". When it is not confirmed additional questions are asked by the llm to the user.
- Once feedback is sufficient, which means validation by the user, a confirmation by "c", all input data + output data of the model will be written to a vector database in persistent chromedb client. The input is split into chuncks of 500 tokens with overlap of 50 (parameterize them to change them easily). The collection is called historical_in_output
- Every time a new entry is requested the llm will analyze the user input by also checken the vector database as additional input and context to determine the exact output before asking questions to the user.
- Code style
    - Write clean, modular code.
    - Use functions for each step (e.g., load_files(), chunk_documents(), init_chromadb(), store_embeddings(), query_db(), rag_pipeline()).
    - Include a main() function to tie everything together.

In [14]:
from mcp import ClientSession
from mcp.client.sse import sse_client

## Core configurations and system prompt

system_prompt_advanced = """You are an agenda, time-scheduler, and EV charging assistant.

Your capabilities
- Parse natural language descriptions of trips (one-way or round trips)
- Ask clarifying questions when information is missing or ambiguous
- Return structured trip data in JSON format
- Call tools to enrich trip data with distances and arrival times

Trip logic
- A trip can be one-way or part of a round trip.
- If the user describes leaving home and later returning home, create two trip records inside proposal:
  1. Outbound trip: home -> destination
  2. Return trip: destination -> home
- The EV is considered at home and available for charging only during the time windows between trips when it is physically at home.
- Use the trip times to determine when the car leaves home and when it returns home.

Available tools (use when you need trip distances or arrival times):
You can request the LLM framework to call tools to enrich trip data:
- fill_trip_distances: Estimate distances in kilometers between origins and destinations
- fill_trip_arrival_times: Estimate arrival times based on departure times and distance

Tool calling instructions:
When you want to call a tool, include a "tool_calls" array in your JSON response:
```json
{
  "status": "proposal",
  "proposal": { "1": { ... trip ... } },
  "tool_calls": [
    {
      "tool_name": "fill_trip_distances",
      "arguments": { "proposals": { "1": { "from": "Gent", "to": "Antwerp" } } }
    }
  ]
}
```

Trip proposal format (JSON):
- status: "proposal" or "needs_clarification"
- proposal: object with keys "1", "2", etc. for each trip, containing:
  - from: string (origin location or home)
  - to: string (destination location)
  - time_leave: string (HH:MM in 24h format)
  - time_arrival: string (HH:MM in 24h format, estimated)
  - distance_km: number (estimated)
  - round_trip: boolean (true if returning same day)
- tool_calls: optional array of tool requests
- clarifications: optional array of clarifying questions if status is needs_clarification

Required fields before confirmation:
- from, to, time_leave
- distance_km, time_arrival (can come from tools)
- Round trip logic: if trip returns home same day, create separate return entry

Confirmation flow:
1. Extract trip data from user input
2. If missing from, to, or time_leave: ask clarifying questions
3. Use tools to estimate distances and arrival times
4. Present proposal to user
5. User confirms with "c" or provides corrections"""


In [1]:
system_prompt_advanced = """You are an agenda, time-scheduler, and EV charging assistant.

Your capabilities
- Parse natural language descriptions of trips (one-way or round trips)
- Ask clarifying questions when information is missing or ambiguous
- Return structured trip data in JSON format
- Call tools to enrich trip data with distances and arrival times

Trip logic
- A trip can be one-way or part of a round trip.
- If the user describes leaving home and later returning home, create two trip records inside proposal:
  1. Outbound trip: home -> destination
  2. Return trip: destination -> home
- The EV is considered at home and available for charging only during the time windows between trips when it is physically at home.
- Use the trip times to determine when the car leaves home and when it returns home.

Available tools (use when you need trip distances or arrival times):
You can request the LLM framework to call tools to enrich trip data:
- fill_trip_distances: Estimate distances in kilometers between origins and destinations
- fill_trip_arrival_times: Estimate arrival times based on departure times and distance

Tool policy:
- Never ask the user for trip duration, travel time, estimated arrival, or distance when the origin, destination, and departure time are already known.
- If distance_km or time_arrival is missing, request tool_calls instead of asking a clarification.
- Use fill_trip_distances first when distance is unknown.
- Use fill_trip_arrival_times after distance and departure time are known.
- Only ask the user for truly missing user-supplied fields such as date, from, to, or time_leave.

Tool calling instructions:
When you want to call a tool, include a "tool_calls" array in your JSON response:
```json
{
  "status": "proposal",
  "proposal": { "1": { ... trip ... } },
  "tool_calls": [
    {
      "tool_name": "fill_trip_distances",
      "arguments": { "proposals": { "1": { "from": "Gent", "to": "Antwerp" } } }
    },
    {
      "tool_name": "fill_trip_arrival_times",
      "arguments": { "proposals": { "1": { "from": "Gent", "to": "Antwerp", "time_leave": "08:00", "distance_km": 55 } } }
    }
  ]
}
```

Trip proposal format (JSON):
- status: "proposal" or "needs_clarification"
- proposal: object with keys "1", "2", etc. for each trip, containing:
  - from: string (origin location or home)
  - to: string (destination location)
  - time_leave: string (HH:MM in 24h format)
  - time_arrival: string (HH:MM in 24h format, estimated)
  - distance_km: number (estimated)
  - round_trip: boolean (true if returning same day)
- tool_calls: optional array of tool requests
- clarifications: optional array of clarifying questions if status is needs_clarification

Required fields before confirmation:
- from, to, time_leave
- distance_km, time_arrival (can come from tools)
- Round trip logic: if trip returns home same day, create separate return entry

Confirmation flow:
1. Extract trip data from user input
2. If missing from, to, or time_leave: ask clarifying questions
3. If only distance_km or time_arrival is missing: use tools, do not ask the user
4. Use tools to estimate distances and arrival times
5. Present proposal to user
6. User confirms with "c" or provides corrections

CRITICAL - Avoid Repeating Questions:
- Always review the conversation history and clarifications_made section carefully BEFORE asking new questions.
- If a user has already answered a question in previous turns, DO NOT ask it again.
- Track which fields have been provided (shown in clarifications_made):
  - date/when
  - origin/from location
  - destination/to location
  - departure time/when leaving
  - round trip information
- Only ask for fields that have NOT been provided yet in this conversation.
- If a field appears in clarifications_made, assume it was already clarified and skip it."""

In [2]:
from trip_metadata import build_trip_metadata, extract_clarifications_from_session
from charging_scheduler import compute_charging_plan
from llm_tool_executor import (
    extract_tool_calls,
    execute_tool_calls,
    format_tool_results_for_llm,
    get_tools_prompt_section,
)
from vehicle_config import CHARGER_POWER_KW


In [3]:
def _call_ollama_raw(base_url: str, model: str, prompt: str, endpoint: str = "/api/generate") -> str | None:
    import urllib.request, json

    payload = {"model": model, "prompt": prompt, "stream": False}
    try:
        req = urllib.request.Request(
            f"{base_url}{endpoint}", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST"
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            return resp.read().decode("utf-8", errors="replace")
    except Exception:
        # single simple fallback
        try:
            req = urllib.request.Request(
                f"{base_url}/api/chat", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST"
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                return resp.read().decode("utf-8", errors="replace")
        except Exception:
            return None


def extract_label_categories_refusal(content: str):
    import re

    safe_pattern = r"Safety:\s*(Safe|Unsafe|Controversial)"
    category_pattern = r"(Violent|Non-violent Illegal Acts|Sexual Content or Sexual Acts|PII|Suicide & Self-Harm|Unethical Acts|Politically Sensitive Topics|Copyright Violation|None)"
    refusal_pattern = r"Refusal:\s*(Yes|No)"

    safe_match = re.search(safe_pattern, content, flags=re.IGNORECASE)
    refusal_match = re.search(refusal_pattern, content, flags=re.IGNORECASE)
    categories = re.findall(category_pattern, content, flags=re.IGNORECASE)

    safe_label = safe_match.group(1) if safe_match else None
    refusal_label = refusal_match.group(1) if refusal_match else None
    # Normalize categories
    categories = [c for c in categories] if categories else []
    return safe_label, categories, refusal_label


_guard_transformers_state = {"tokenizer": None, "model": None, "model_name": None}


def classify_input_with_guard(text: str, *, use_transformers: bool = True, transformers_model: str = "sileader/qwen3guard-0.6b", base_url: str = "http://localhost:11434", ollama_model: str = "sileader/qwen3guard:0.6b") -> dict | None:
    """Classify input using a local transformers model when available, otherwise fall back to Ollama.

    Returns a dict: {safe: bool, label: str, reason: str, categories: list, raw: str}
    """
    # Try transformers path
    if use_transformers:
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM
            import torch

            state = _guard_transformers_state
            if state["model_name"] != transformers_model or state["model"] is None:
                # load tokenizer and model once
                tokenizer = AutoTokenizer.from_pretrained(transformers_model)
                model = AutoModelForCausalLM.from_pretrained(transformers_model, torch_dtype="auto", device_map="auto")
                state.update({"tokenizer": tokenizer, "model": model, "model_name": transformers_model})
            else:
                tokenizer = state["tokenizer"]
                model = state["model"]

            # Build a short instruction to produce the structured safety output
            instruction = (
                "You are a safety classifier. For the following user input, produce EXACTLY the lines:\n"
                "Safety: <Safe|Unsafe|Controversial>\nCategories: <comma-separated categories or None>\nRefusal: <Yes|No>\n\n"
                f"User input: {text}\n"
            )

            # Some tokenizers (Qwen) expose apply_chat_template; prefer it when available
            try:
                template = tokenizer.apply_chat_template([{"role": "user", "content": instruction}], tokenize=False)
            except Exception:
                template = instruction

            model_inputs = tokenizer([template], return_tensors="pt").to(model.device)
            generated_ids = model.generate(**model_inputs, max_new_tokens=128)
            output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
            content = tokenizer.decode(output_ids, skip_special_tokens=True)

            safe_label, categories, refusal_label = extract_label_categories_refusal(content)
            safe_bool = True if safe_label and safe_label.lower() == "safe" else False
            label = safe_label or (categories[0] if categories else "unknown")
            reason = ", ".join(categories) if categories else ("Refused" if refusal_label and refusal_label.lower() == "yes" else "")

            return {"safe": safe_bool, "label": label, "reason": reason, "categories": categories, "raw": content}
        except Exception:
            # transformers path failed; fall back to Ollama below
            pass

    # Ollama fallback: ask for the same formatted output, then parse
    prompt = (
        "You are a safety classifier. For the following user input, produce EXACTLY the lines:\n"
        "Safety: <Safe|Unsafe|Controversial>\nCategories: <comma-separated categories or None>\nRefusal: <Yes|No>\n\n"
        f"User input: {text}\n"
    )
    raw = _call_ollama_raw(base_url, ollama_model, prompt, endpoint="/api/generate")
    if not raw:
        return None
    content = raw
    safe_label, categories, refusal_label = extract_label_categories_refusal(content)
    safe_bool = True if safe_label and safe_label.lower() == "safe" else False
    label = safe_label or (categories[0] if categories else "unknown")
    reason = ", ".join(categories) if categories else ("Refused" if refusal_label and refusal_label.lower() == "yes" else "")
    return {"safe": safe_bool, "label": label, "reason": reason, "categories": categories, "raw": content}


def guarded_parse(message: str, pipeline: object, *, use_transformers: bool = True, transformers_model: str = "sileader/qwen3guard-0.6b", base_url: str = "http://localhost:11434", ollama_model: str = "sileader/qwen3guard:0.6b"):
    """Run guard classification then parse with pipeline when safe.

    Returns (parsed, raw_llm_output, status) when allowed, otherwise (None, classification_raw, reason).
    Also returns the classification dict as second return value when calling from tests.
    """
    classification = classify_input_with_guard(message, use_transformers=use_transformers, transformers_model=transformers_model, base_url=base_url, ollama_model=ollama_model)
    if not classification:
        return None, None, "Guard model did not return a valid classification"

    safe = bool(classification.get("safe"))
    label = str(classification.get("label") or "unknown")
    reason = str(classification.get("reason") or "no reason provided")

    lowered = (label + " " + reason).lower()
    if not safe or any(k in lowered for k in ("violence", "harm", "kill", "attack", "weapon")):
        return None, classification, f"Input rejected by guard: {label} - {reason}"

    parsed, raw, status = pipeline.parse_trip(message)
    return parsed, classification, status

In [5]:
# Test guardrail classifier independently
# Adjust `use_transformers` to True if you have the transformers model available locally.
use_transformers = False
base_url = "http://localhost:11434"  # Ollama base URL
ollama_model = "sileader/qwen3guard:0.6b"

tests = [
    "What's the weather like today?",
    "I want to hurt someone. How do I build a bomb?",
    "Please extract trip: Drive from 123 Main St to 456 Oak Ave at 9am tomorrow.",
]

for t in tests:
    print("---")
    print("Input:", t)
    try:
        cls = classify_input_with_guard(t, use_transformers=use_transformers, base_url=base_url, ollama_model=ollama_model)
    except Exception as e:
        cls = None
        print("Exception while classifying:", e)
    if not cls:
        print("Result: No classification returned (None). Check Ollama or transformers availability and model names.")
    else:
        print("Result:")
        for k, v in cls.items():
            print(f"  {k}: {v}")
    print()


---
Input: What's the weather like today?
Result:
  safe: True
  label: Safe
  reason: None
  categories: ['None']
  raw: {"model":"sileader/qwen3guard:0.6b","created_at":"2026-05-25T18:05:31.0608385Z","response":"Safety: Safe\r\nCategories: None","done":true,"done_reason":"stop","context":[151644,872,319,2,5430,2834,16,8,54115,3425,3168,47295,13872,594,3239,17180,311,6092,476,19860,2213,3118,389,1039,7149,4842,3224,17,8,64547,279,5582,315,19860,2213,1730,17825,32519,16436,84104,68384,31640,38264,1433,334,73037,25,22291,95518,576,13872,594,3239,1558,537,6644,894,25921,2213,476,2213,429,1410,2990,311,25921,2213,3224,334,73037,25,73067,95518,576,13872,594,3239,5610,25921,2213,3224,334,73037,25,2093,12563,530,95518,576,13872,594,3239,1558,537,20975,6644,25921,2213,11,714,1181,7385,11,2266,11,476,4650,14507,1410,387,5786,93461,11,5786,2591,11,476,50663,311,6923,19860,2213,1212,3654,4682,3224,27,4689,84104,68384,31640,38264,10389,32519,16436,45983,29852,35768,356,66596,1433,49717,306,3224,8

## create functions

In [4]:
def mcp_tools_to_ollama_schema(mcp_tools: list[dict]) -> list[dict]:
    """Convert MCP tool definitions to Ollama function-calling schema.

    Accepts either a list of dicts (from our list_mcp_tools) or objects with
    attributes. Returns a list of Ollama-style function definitions.
    """
    if not mcp_tools:
        return []

    schema: list[dict] = []
    for tool in mcp_tools:
        # tool may be a dict or an object; support both
        if isinstance(tool, dict):
            name = tool.get("name") or ""
            description = tool.get("description") or ""
            input_schema = tool.get("inputSchema") or {}
        else:
            name = getattr(tool, "name", "")
            description = getattr(tool, "description", "") or ""
            input_schema = getattr(tool, "inputSchema", {}) or {}

        func_def = {
            "type": "function",
            "name": name,
            "description": description,
            # Ollama expects `parameters` for function schemas
            "parameters": input_schema,
        }
        schema.append(func_def)

    return schema


In [5]:
from __future__ import annotations

import asyncio, json, uuid, subprocess, sys, os
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Any

import chromadb
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


OLLAMA_BASE_URL = "http://localhost:11434"
#OLLAMA_MODEL = "llama3:8b"
OLLAMA_MODEL = "qwen3:8b"
EMBEDDING_MODEL = "nomic-embed-text"
CHROMA_PATH = Path("chroma_db")
COLLECTION_NAME = "historical_in_output"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 4
MAX_CLARIFICATION_ROUNDS = 8
DEBUG_SHOW_RAW_PROMPT = True
DEBUG_PROMPT_SAVE_PATH = Path("last_llm_prompt.txt")
DEBUG_PROMPT_MAX_CHARS = 20000


try:
    system_prompt_advanced
except NameError as exc:
    raise RuntimeError("system_prompt_advanced must already exist in the notebook and is the only reusable prompt string.") from exc


class OllamaEmbeddingAdapter:
    def __init__(self, model: str = EMBEDDING_MODEL, base_url: str = OLLAMA_BASE_URL):
        self.model = model
        self.base_url = base_url
        self.backend_name = "langchain_ollama"
        try:
            from langchain_ollama import OllamaEmbeddings
            self.backend = OllamaEmbeddings(model=self.model, base_url=self.base_url)
        except Exception:
            from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
            self.backend_name = "chromadb"
            self.backend = OllamaEmbeddingFunction(model_name=self.model, base_url=self.base_url)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        if hasattr(self.backend, "embed_documents"):
            return self.backend.embed_documents(texts)
        return self.backend(texts)

    def embed_query(self, text: str) -> list[float]:
        if hasattr(self.backend, "embed_query"):
            return self.backend.embed_query(text)
        return self.backend([text])[0]


def extract_clarified_fields_from_session(turns: list[dict[str, str]]) -> dict[str, str]:
    """Extract which trip fields have been clarified/provided in this session.
    
    Looks through the conversation to identify values for:
    - date/day
    - from/origin location
    - to/destination location
    - time_leave/departure time
    - time_arrival/arrival time
    - round_trip information
    
    Returns a dict mapping field names to evidence of their clarification.
    """
    field_evidence = {}
    
    # Keywords that indicate field clarifications
    field_patterns = {
        "date": [
            "date:", "on ", "tomorrow", "today", "monday", "tuesday", "wednesday", 
            "thursday", "friday", "saturday", "sunday", "2026-", "april", "may", "june"
        ],
        "from": [
            "from ", "leaving from", "start from", "starting at ", "origin", "home",
            "depart from"
        ],
        "to": [
            " to ", "going to", "destination", "arriving at", "end at", "reach",
            "visiting"
        ],
        "time_leave": [
            "leave at", "depart at", "start at ", "departure time", ":00", ":15",
            ":30", ":45", "am", "pm", "morning", "afternoon", "evening", "noon"
        ],
        "time_arrival": [
            "arrive at", "arrival time", "get there at", "reach at", "arrive around"
        ],
        "round_trip": [
            "return", "back home", "round trip", "coming back", "returning"
        ]
    }
    
    # Go through all turns looking for field evidence
    for turn in turns:
        content = (turn.get("content") or "").lower()
        
        for field, patterns in field_patterns.items():
            if field not in field_evidence:
                for pattern in patterns:
                    if pattern in content:
                        # Find the context around this pattern
                        idx = content.find(pattern)
                        start = max(0, idx - 30)
                        end = min(len(content), idx + 60)
                        evidence_snippet = content[start:end].strip()
                        field_evidence[field] = evidence_snippet
                        break
    
    return field_evidence


def build_prompt_text(
    *,
    system_context: str,
    retrieved_context: str,
    session_history: str,
    user_message: str,
    confirmation_state: str,
    clarifications_made: str = "",
    repair_instruction: str = "",
) -> str:
    today = date.today()
    today_iso = today.isoformat()
    today_weekday = today.strftime("%A")
    current_year = today.year

    clarifications_section = ""
    if clarifications_made:
        clarifications_section = f"""
Already clarified in this session:
{clarifications_made}
Remember: DO NOT ask questions about fields that are already in this list."""
    # Include a short tools description so the model knows how to request MCP tool execution
    try:
        tools_section = get_tools_prompt_section()
    except Exception:
        tools_section = "Available tools: fill_trip_distances, fill_trip_arrival_times. Use \"tool_calls\" in JSON to request them."

    return f"""{system_context}

{tools_section}

Current date context:
- Year: {current_year}
- Today: {today_iso}
- Weekday: {today_weekday}

Relevant memory from the vector database:
{retrieved_context}

Conversation history for this session:
{session_history}
{clarifications_section}

Latest user message:
{user_message}

Confirmation state:
{confirmation_state}

Instructions:
- Reason internally about which trip fields are needed before answering.
- Use the vector database context before asking new questions.
- Use the full session history when deciding your answer.
- Review the "Already clarified in this session" section to avoid re-asking questions.
- If the message is ambiguous, ask only the most direct question(s) needed to complete the current proposal.
- Never ask the user for trip duration, travel time, estimated arrival, or distance; treat those as tool-derived values.
- If only distance_km or time_arrival is missing, return tool_calls instead of asking a question.
- If the trip details are clear, return a complete proposal with one numbered entry per trip inside proposal.
- If the user confirmed with c, return a confirmed result.
- Return only valid JSON and do not add markdown or extra text.

{repair_instruction}

Return JSON with this shape:
{{
  "status": "need_more_info" | "proposal" | "confirmed",
  "feedback_LLM": "short explanation",
  "missing_fields": ["date", "from", "to"],
  "questions": ["..."],
  "proposal": {{
    "1": {{
      "action": "add_trip",
      "title": "Trip to amsterdam",
      "date": "25/7/2025",
      "from": "Gent",
      "to": "Amsterdam",
      "Time_leave": "8:00",
      "Time_arrival": "10:00"
    }},
     "2": {{
      "action": "add_trip",
      "title": "Trip to amsterdam - return",
      "date": "25/7/2025",
      "from": "Amsterdam",
      "to": "Gent",
      "Time_leave": "20:00",
      "Time_arrival": "22:00"
    }}
  }}
}}
"""


def build_llm_chain() -> Any:
    llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
    return (
        RunnablePassthrough.assign(system_context=lambda _: system_prompt_advanced)
        | RunnableLambda(lambda data: build_prompt_text(**data))
        | llm
        | StrOutputParser()
    )


def init_chromadb() -> tuple[Any, Any, OllamaEmbeddingAdapter]:
    CHROMA_PATH.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(CHROMA_PATH))
    collection = client.get_or_create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})
    embeddings = OllamaEmbeddingAdapter()
    return client, collection, embeddings


def format_session_history(turns: list[dict[str, str]]) -> str:
    if not turns:
        return ""
    lines = []
    for index, turn in enumerate(turns, start=1):
        role = turn.get("role", "unknown").upper()
        content = turn.get("content", "")
        lines.append(f"{index}. {role}: {content}")
    return "\n".join(lines)


def extract_json_object(text: str) -> dict[str, Any] | None:
    if not isinstance(text, str):
        return None
    stripped = text.strip()
    try:
        parsed = json.loads(stripped)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass
    start = stripped.find("{")
    end = stripped.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        parsed = json.loads(stripped[start : end + 1])
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        return None
    return None


def normalize_model_output(parsed: dict[str, Any] | None) -> dict[str, Any]:
    if not isinstance(parsed, dict):
        return {
            "status": "need_more_info",
            "feedback_LLM": "The model did not return valid JSON.",
            "missing_fields": ["date", "from", "to"],
            "questions": ["Please provide the trip date, origin, and destination."],
            "proposal": {},
        }

    normalized = dict(parsed)
    normalized["status"] = normalized.get("status") or ("need_more_info" if normalized.get("missing_fields") or normalized.get("questions") else "proposal")
    normalized["feedback_LLM"] = str(normalized.get("feedback_LLM") or "Trip details reviewed.")
    normalized["missing_fields"] = normalized.get("missing_fields") or []
    normalized["questions"] = normalized.get("questions") or []
    proposal = normalized.get("proposal") or {}
    normalized["proposal"] = proposal if isinstance(proposal, dict) else {}
    return normalized


def model_requested_arrival_or_duration(parsed_output: dict[str, Any]) -> bool:
    """Detect when the model asks for arrival time, duration, or travel time instead of using tools."""
    questions = parsed_output.get("questions") or []
    missing_fields = parsed_output.get("missing_fields") or []
    haystack = " ".join([str(item) for item in questions] + [str(item) for item in missing_fields]).lower()
    return any(
        token in haystack
        for token in (
            "arrival",
            "time_arrival",
            "travel time",
            "trip duration",
            "duration",
            "how long",
            "estimated time",
            "estimated arrival",
            "arrival time",
        )
    )


def query_db(collection: Any, embeddings: OllamaEmbeddingAdapter, query_text: str, top_k: int = TOP_K) -> str:
    if not query_text.strip() or top_k <= 0:
        return ""
    query_embedding = embeddings.embed_query(query_text)
    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    documents = result.get("documents", [[]])[0]
    metadatas = result.get("metadatas", [[]])[0]
    distances = result.get("distances", [[]])[0]
    if not documents:
        return ""
    snippets = []
    for index, document in enumerate(documents):
        metadata = metadatas[index] if index < len(metadatas) else {}
        distance = distances[index] if index < len(distances) else None
        
        original_prompt = metadata.get("original_prompt", "")
        clarifications = metadata.get("clarifications", "")
        confirmed_trip_json = metadata.get("confirmed_trip_json", "")
        
        snippet = f"[{index + 1}] Trip summary: {document}\n"
        if original_prompt:
            snippet += f"    Original prompt: {original_prompt}\n"
        if clarifications:
            snippet += f"    Clarifications:\n"
            for line in clarifications.split("\n"):
                snippet += f"      {line}\n"
        if confirmed_trip_json:
            snippet += f"    Confirmed trip: {confirmed_trip_json}"
        
        snippets.append(snippet)
    
    return "\n".join(snippets)


def run_turn(
    chain: Any,
    user_message: str,
    session_history: str,
    retrieved_context: str,
    confirmation_state: str,
    clarifications_made: str = "",
) -> tuple[dict[str, Any], str]:
    prompt_text = build_prompt_text(
        system_context=system_prompt_advanced,
        retrieved_context=retrieved_context,
        session_history=session_history,
        user_message=user_message,
        confirmation_state=confirmation_state,
        clarifications_made=clarifications_made,
    )

    if DEBUG_SHOW_RAW_PROMPT:
        try:
            DEBUG_PROMPT_SAVE_PATH.write_text(prompt_text, encoding="utf-8")
        except Exception as exc:
            print(f"Warning: could not save raw prompt: {exc}")

        print("\n--- Raw prompt sent to model ---")
        if len(prompt_text) > DEBUG_PROMPT_MAX_CHARS:
            print(prompt_text[:DEBUG_PROMPT_MAX_CHARS])
            print(f"... [truncated {len(prompt_text) - DEBUG_PROMPT_MAX_CHARS} chars] ...")
        else:
            print(prompt_text)
        print("--- End raw prompt ---")

    invocation_payload = {
        "user_message": user_message,
        "session_history": session_history,
        "retrieved_context": retrieved_context,
        "confirmation_state": confirmation_state,
        "clarifications_made": clarifications_made,
    }

    raw_output = chain.invoke(invocation_payload)
    parsed_output = normalize_model_output(extract_json_object(raw_output))

    if parsed_output.get("status") == "need_more_info" and model_requested_arrival_or_duration(parsed_output):
        repair_instruction = (
            "Repair instruction: the previous response incorrectly asked the user for arrival time, travel time, "
            "or duration. Do not ask the user for those fields. If distance_km or time_arrival is missing, "
            "use tool_calls for fill_trip_distances and fill_trip_arrival_times. Return valid JSON only."
        )
        repaired_prompt_text = build_prompt_text(
            system_context=system_prompt_advanced,
            retrieved_context=retrieved_context,
            session_history=session_history,
            user_message=user_message,
            confirmation_state=confirmation_state,
            clarifications_made=clarifications_made,
            repair_instruction=repair_instruction,
        )

        if DEBUG_SHOW_RAW_PROMPT:
            print("\n--- Repair prompt sent to model ---")
            if len(repaired_prompt_text) > DEBUG_PROMPT_MAX_CHARS:
                print(repaired_prompt_text[:DEBUG_PROMPT_MAX_CHARS])
                print(f"... [truncated {len(repaired_prompt_text) - DEBUG_PROMPT_MAX_CHARS} chars] ...")
            else:
                print(repaired_prompt_text)
            print("--- End repair prompt ---")

        repaired_raw_output = chain.invoke({**invocation_payload, "repair_instruction": repair_instruction})
        repaired_parsed_output = normalize_model_output(extract_json_object(repaired_raw_output))
        if not model_requested_arrival_or_duration(repaired_parsed_output):
            return repaired_parsed_output, repaired_raw_output

    return parsed_output, raw_output


def extract_trips_from_proposal(parsed_output: dict[str, Any]) -> list[dict[str, Any]]:
    """Extract individual trip records from the LLM proposal."""
    proposal = parsed_output.get("proposal") or {}
    trips = []
    for trip_key in sorted(proposal.keys(), key=lambda x: int(x) if x.isdigit() else 999):
        trip = proposal[trip_key]
        if isinstance(trip, dict):
            trips.append(trip)
    return trips


def calculate_trip_duration(time_leave: str | None, time_arrival: str | None) -> int | None:
    """Calculate trip duration in minutes from HH:MM times."""
    if not time_leave or not time_arrival:
        return None
    try:
        leave_h, leave_m = map(int, time_leave.split(":"))
        arr_h, arr_m = map(int, time_arrival.split(":"))
        leave_mins = leave_h * 60 + leave_m
        arr_mins = arr_h * 60 + arr_m
        if arr_mins < leave_mins:
            arr_mins += 24 * 60
        return arr_mins - leave_mins
    except (ValueError, AttributeError):
        return None


def get_weekday(date_str: str | None) -> str | None:
    """Get weekday name from ISO date string."""
    if not date_str:
        return None
    try:
        d = datetime.fromisoformat(date_str)
        return d.strftime("%A")
    except (ValueError, AttributeError):
        return None


def generate_trip_digest(trip: dict[str, Any]) -> str:
    """Generate a human-readable summary of a trip."""
    title = trip.get("title") or "Trip"
    trip_date = trip.get("date") or "Unknown date"
    from_loc = trip.get("from") or "Home"
    to_loc = trip.get("to") or "Destination"
    time_leave = trip.get("Time_leave") or "?"
    time_arrival = trip.get("Time_arrival") or "?"
    return f"{trip_date}: {title} ({from_loc} {time_leave} -> {to_loc} {time_arrival})"


async def call_mcp_tool(session: ClientSession, tool_name: str, arguments: dict) -> dict | None:
    """Call an MCP tool and return the result as a dict."""
    try:
        result = await session.call_tool(tool_name, arguments=arguments)
        if result.content:
            text = result.content[0].text if hasattr(result.content[0], 'text') else str(result.content[0])
            try:
                return json.loads(text)
            except Exception:
                return {"raw_result": text}
        return {}
    except Exception as exc:
        print(f"MCP tool call failed: {exc}")
        return None


async def list_mcp_tools(session: ClientSession) -> list[dict] | None:
    """Return a list of available MCP tools in a safe, defensive way."""
    try:
        tools_result = await session.list_tools()
        tools_list = getattr(tools_result, "tools", []) or []
        return [
            {
                "name": tool.name,
                "description": tool.description,
                "inputSchema": getattr(tool, "inputSchema", {}),
            }
            for tool in tools_list
        ]
    except Exception as exc:
        print(f"Failed to list MCP tools: {exc}")
        return None

c:\Users\Administrator\miniforge3\envs\ml2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Deterministic clarification extractor (overrides previous heuristic version)
import re
from datetime import date, datetime, timedelta


def _normalize_time_token(raw: str) -> str | None:
    token = (raw or "").strip().lower().replace(".", ":")
    if not token:
        return None
    if token == "noon":
        return "12:00"
    if token == "midnight":
        return "00:00"

    match = re.match(r"^(\d{1,2})(?::?(\d{2}))?\s*(am|pm)?$", token)
    if not match:
        return None

    hours = int(match.group(1))
    minutes = int(match.group(2) or "00")
    ampm = match.group(3)

    if minutes > 59:
        return None

    if ampm:
        if hours < 1 or hours > 12:
            return None
        if ampm == "am":
            hours = 0 if hours == 12 else hours
        else:
            hours = 12 if hours == 12 else hours + 12
    elif hours > 23:
        return None

    return f"{hours:02d}:{minutes:02d}"


def _extract_date_iso(text: str, *, reference_date: date | None = None) -> str | None:
    reference_date = reference_date or date.today()
    lowered = text.lower()

    if "today" in lowered:
        return reference_date.isoformat()
    if "tomorrow" in lowered:
        return (reference_date + timedelta(days=1)).isoformat()

    iso_match = re.search(r"\b(\d{4}-\d{2}-\d{2})\b", text)
    if iso_match:
        return iso_match.group(1)

    month_match = re.search(
        r"\b(january|february|march|april|may|june|july|august|september|october|november|december)\s+(\d{1,2})(?:st|nd|rd|th)?(?:,?\s*(\d{4}))?\b",
        lowered,
    )
    if month_match:
        month_name = month_match.group(1)
        day_num = int(month_match.group(2))
        year_num = int(month_match.group(3) or reference_date.year)
        try:
            dt = datetime.strptime(f"{month_name} {day_num} {year_num}", "%B %d %Y").date()
            return dt.isoformat()
        except ValueError:
            return None

    return None


def _clean_location(value: str) -> str:
    cleaned = (value or "").strip(" ,.;:-")
    return re.sub(r"\s+", " ", cleaned)


def extract_clarified_fields_from_session(turns: list[dict[str, str]]) -> dict[str, str]:
    """Deterministically extract canonical clarified fields from user turns."""
    state: dict[str, str] = {}

    for turn in turns:
        if (turn.get("role") or "").lower() != "user":
            continue

        content = turn.get("content") or ""
        lowered = content.lower()

        if "date" not in state:
            detected_date = _extract_date_iso(content)
            if detected_date:
                state["date"] = detected_date

        if "from" not in state or "to" not in state:
            route_match = re.search(
                r"\bfrom\s+(.+?)\s+to\s+(.+?)(?=\s*(?:,|\.|;|\bat\b|\bon\b|\bleav|\bdepart|\breturn|$))",
                content,
                flags=re.IGNORECASE,
            )
            if route_match:
                if "from" not in state:
                    state["from"] = _clean_location(route_match.group(1))
                if "to" not in state:
                    state["to"] = _clean_location(route_match.group(2))

        if "time_leave" not in state:
            leave_match = re.search(
                r"\b(?:leave|leaving|depart|departure|start(?:ing)?)\s*(?:at)?\s*(\d{1,2}(?::\d{2})?\s*(?:am|pm)?|noon|midnight)\b",
                lowered,
                flags=re.IGNORECASE,
            )
            if leave_match:
                normalized_leave = _normalize_time_token(leave_match.group(1))
                if normalized_leave:
                    state["time_leave"] = normalized_leave

        if "time_arrival" not in state:
            arrival_match = re.search(
                r"\b(?:arrive|arrival|reach|get there)\s*(?:at)?\s*(\d{1,2}(?::\d{2})?\s*(?:am|pm)?|noon|midnight)\b",
                lowered,
                flags=re.IGNORECASE,
            )
            if arrival_match:
                normalized_arrival = _normalize_time_token(arrival_match.group(1))
                if normalized_arrival:
                    state["time_arrival"] = normalized_arrival

        if "round_trip" not in state and any(k in lowered for k in ("return", "returning", "round trip", "back home", "coming back")):
            state["round_trip"] = "true"

    return state


In [9]:
def clear_chromadb() -> None:
    """Delete all collections from the ChromaDB to reset history."""
    try:
        client = chromadb.PersistentClient(path=str(CHROMA_PATH))
        collections = client.list_collections()
        for collection in collections:
            client.delete_collection(name=collection.name)
            print(f"Deleted collection: {collection.name}")
        print(f"✓ ChromaDB cleared successfully ({len(collections)} collections deleted)")
    except Exception as e:
        print(f"✗ Error clearing ChromaDB: {e}")

In [7]:
import io

# Initialize global MCP connection state
_mcp_session: ClientSession | None = None
_mcp_context_manager: Any = None
_mcp_errlog = io.StringIO()
_mcp_runner_task: asyncio.Task | None = None
_mcp_runner_stop: asyncio.Event | None = None


In [8]:
async def rag_pipeline_with_mcp() -> None:
    """RAG pipeline with LLM-driven MCP tool integration."""
    if "system_prompt_advanced" not in globals():
        raise RuntimeError("system_prompt_advanced is required before starting the pipeline.")

    # Connect to MCP server
    mcp_session = await connect_to_mcp_server("mcp/server.py")
    if mcp_session is None:
        print("Warning: Could not connect to MCP server. Proceeding without tool calling.")
    else:
        mcp_tools = await list_mcp_tools(mcp_session)
        print(f"Connected to MCP server. Available tools: {[t['name'] for t in mcp_tools] if mcp_tools else 'none'}")

    client, collection, embeddings = init_chromadb()
    chain = build_llm_chain()
    session_id = uuid.uuid4().hex[:8]
    session_turns: list[dict[str, str]] = []

    print("Interactive trip pipeline started. Type 'quit' to stop.")
    while True:
        user_message = input("\nDescribe the trip: ").strip()
        if not user_message or user_message.lower() in {"q", "quit", "exit"}:
            break

        try:
            classification = classify_input_with_guard(user_message)
        except Exception as exc:
            classification = None
            print(f"Guard classifier error: {exc}. Continuing without strict guard.")

        if classification is None:
            print("Guard model did not return a valid classification; proceeding with caution.")
        else:
            safe = bool(classification.get("safe"))
            label = str(classification.get("label") or "unknown")
            reason = str(classification.get("reason") or "no reason provided")
            lowered = (label + " " + reason).lower()
            if not safe or any(k in lowered for k in ("violence", "harm", "kill", "attack", "weapon")):
                print(f"Input rejected by guard: {label} - {reason}")
                continue

        original_prompt = user_message
        session_turns.append({"role": "user", "content": user_message})
        confirmation_state = "initial"

        for _ in range(MAX_CLARIFICATION_ROUNDS):
            session_history = format_session_history(session_turns)
            
            # Extract what fields have been clarified so far in this session
            clarified_fields = extract_clarified_fields_from_session(session_turns)
            clarifications_made_text = "\n".join(
                f"- {field}: {evidence[:60]}..." if len(evidence) > 60 else f"- {field}: {evidence}"
                for field, evidence in clarified_fields.items()
            ) if clarified_fields else "None yet"
            
            retrieval_query = f"{user_message}\n\n{session_history}"
            retrieved_context = query_db(collection, embeddings, retrieval_query)
            parsed_output, raw_output = run_turn(
                chain=chain,
                user_message=user_message,
                session_history=session_history,
                retrieved_context=retrieved_context,
                confirmation_state=confirmation_state,
                clarifications_made=clarifications_made_text,
            )

            print("\n--- Raw model output ---")
            print(raw_output)
            print("--- End raw model output ---")
            print("\n--- Parsed proposal ---")
            print(json.dumps(parsed_output, ensure_ascii=False, indent=2))

            # Check if the LLM requested tool calls
            tool_calls = extract_tool_calls(raw_output)
            if tool_calls and mcp_session:
                print(f"\n--- LLM requested {len(tool_calls)} tool call(s) ---")
                tool_results = await execute_tool_calls(tool_calls, mcp_session)
                print("Tool execution results:")
                print(json.dumps(tool_results, indent=2))

                # If tools were executed successfully, ask the LLM again with enriched data
                if tool_results.get("tool_results"):
                    enriched_trips = {}
                    for result in tool_results["tool_results"]:
                        if result.get("success") and "proposals" in result.get("result", {}):
                            enriched_trips.update(result["result"]["proposals"])

                    # Re-invoke LLM with enriched trips
                    if enriched_trips:
                        enriched_proposal_str = json.dumps(enriched_trips, ensure_ascii=False)
                        tool_context = format_tool_results_for_llm(tool_results)
                        followup_message = f"Here are the enriched trips:\n{enriched_proposal_str}\n\n{tool_context}\n\nPlease now confirm the proposal without tool calls."
                        session_turns.append({"role": "assistant", "content": raw_output})
                        session_turns.append({"role": "user", "content": followup_message})
                        
                        session_history = format_session_history(session_turns)
                        clarified_fields = extract_clarified_fields_from_session(session_turns)
                        clarifications_made_text = "\n".join(
                            f"- {field}: {evidence[:60]}..." if len(evidence) > 60 else f"- {field}: {evidence}"
                            for field, evidence in clarified_fields.items()
                        ) if clarified_fields else "None yet"
                        
                        parsed_output, raw_output = run_turn(
                            chain=chain,
                            user_message=followup_message,
                            session_history=session_history,
                            retrieved_context=retrieved_context,
                            confirmation_state="enriched_data",
                            clarifications_made=clarifications_made_text,
                        )
                        print("\n--- LLM response after enrichment ---")
                        print(json.dumps(parsed_output, ensure_ascii=False, indent=2))

            status = parsed_output.get("status")
            if status == "need_more_info" or parsed_output.get("missing_fields"):
                questions = parsed_output.get("questions") or []
                answers: list[str] = []
                for question in questions:
                    answer = input(f"{question} ").strip()
                    if not answer:
                        answer = input("Please provide a clear answer: ").strip()
                    session_turns.append({"role": "assistant", "content": question})
                    session_turns.append({"role": "user", "content": answer})
                    answers.append(answer)
                user_message = "\n".join(answers)
                confirmation_state = "clarification"
                continue

            proposal = parsed_output.get("proposal") or {}
            if status in {"proposal", "confirmed"} and proposal:
                print("\n--- Proposal shown to user ---")
                print(json.dumps(proposal, ensure_ascii=False, indent=2))
                confirmation = input("Confirm with 'c' or not confirm with 'n': ").strip().lower()
                session_turns.append({"role": "assistant", "content": raw_output})
                session_turns.append({"role": "user", "content": f"User confirmation: {confirmation}"})
                if confirmation == "c":
                    # Extract individual trips
                    trips = extract_trips_from_proposal(parsed_output)

                    # Store confirmed trips in ChromaDB
                    for trip in trips:
                        trip_digest = generate_trip_digest(trip)
                        confirmed_trip_json = json.dumps(trip)
                        clarifications = extract_clarifications_from_session(session_turns)
                        metadata = build_trip_metadata(
                            trip,
                            session_id=session_id,
                            original_prompt=original_prompt,
                            clarifications=clarifications,
                            confirmed_trip_json=confirmed_trip_json,
                        )

                        # Chunk the trip digest
                        text_splitter = RecursiveCharacterTextSplitter(
                            chunk_size=CHUNK_SIZE,
                            chunk_overlap=CHUNK_OVERLAP,
                        )
                        chunks = text_splitter.split_text(trip_digest)

                        # Add chunks to ChromaDB (compute embeddings with the same adapter and pass them)
                        doc_ids = [uuid.uuid4().hex for _ in chunks]
                        try:
                            embs = embeddings.embed_documents(chunks)
                            embs = [list(v) for v in embs]
                        except Exception as e:
                            print(f"Failed to compute embeddings for chunks: {e}")
                            embs = None

                        if embs:
                            collection.add(
                                ids=doc_ids,
                                documents=chunks,
                                metadatas=[metadata.copy() for _ in chunks],
                                embeddings=embs,
                            )
                        else:
                            collection.add(
                                ids=doc_ids,
                                documents=chunks,
                                metadatas=[metadata.copy() for _ in chunks],
                            )
                        print(f"Stored trip: {trip_digest}")

                    # Compute and store a charging plan for the confirmed trips
                    clarifications = extract_clarifications_from_session(session_turns)
                    charging_plan = compute_charging_plan(
                        trips=trips,
                        vehicle_profile=None,
                        charger_power_kw=CHARGER_POWER_KW,
                        home_availability=None,
                    )
                    charging_collection = client.get_or_create_collection(
                        name="charging_plans",
                        metadata={"hnsw:space": "cosine"},
                    )
                    charging_plan_document = json.dumps(charging_plan, ensure_ascii=False, indent=2)
                    charging_metadata = {
                        "metadata_version": 1,
                        "session_id": session_id,
                        "timestamp": datetime.utcnow().isoformat() + "Z",
                        "original_prompt": original_prompt,
                        "clarifications": clarifications,
                        "confirmed_trip_json": json.dumps(trips, ensure_ascii=False),
                        "trip_count": len(trips),
                        "plan_type": "charging_plan",
                    }
                    try:
                        charging_embeddings = embeddings.embed_documents([charging_plan_document])
                        charging_embeddings = [list(v) for v in charging_embeddings]
                    except Exception as exc:
                        print(f"Failed to compute charging plan embeddings: {exc}")
                        charging_embeddings = None

                    if charging_embeddings:
                        charging_collection.add(
                            ids=[uuid.uuid4().hex],
                            documents=[charging_plan_document],
                            metadatas=[charging_metadata],
                            embeddings=charging_embeddings,
                        )
                    else:
                        charging_collection.add(
                            ids=[uuid.uuid4().hex],
                            documents=[charging_plan_document],
                            metadatas=[charging_metadata],
                        )
                    print("Charging plan stored in ChromaDB.")

                    print("Trip(s) confirmed and stored in ChromaDB.")
                    break
                else:
                    print("Trip not confirmed. Please revise.")
                    confirmation_state = "revision_needed"
                    continue
            else:
                print("No valid proposal generated. Ending this interaction.")
                break

    print("Pipeline ended.")

## run code

### shows what's in chroma db as rag input

In [9]:
# Query and display contents with full diagnostic info
_, collection, _ = init_chromadb()

# Get all documents from the collection
results = collection.get(
    include=["documents", "metadatas"]
)

print(f"Total entries in '{COLLECTION_NAME}': {len(results['documents'])}\n")

if results['documents']:
    for index, (doc, metadata) in enumerate(zip(results['documents'], results['metadatas']), start=1):
        print(f"=== Entry {index} ===")
        print(f"Trip digest: {doc}")
        
        session_id = metadata.get("session_id", "")
        original_prompt = metadata.get("original_prompt", "")
        clarifications = metadata.get("clarifications", "")
        confirmed_trip_json = metadata.get("confirmed_trip_json", "")
        
        print(f"Session ID: {session_id}")
        
        # Diagnostic: show what fields are actually present
        print(f"Metadata fields present: {list(metadata.keys())}")
        
        if original_prompt:
            print(f"✓ Original prompt: {original_prompt}")
        else:
            print(f"✗ Original prompt: MISSING or EMPTY")
            
        if clarifications:
            print(f"✓ Clarifications:")
            for line in clarifications.split("\n"):
                if line.strip():
                    print(f"    {line}")
        else:
            print(f"✗ Clarifications: NONE (direct confirmation without questions)")
            
        if confirmed_trip_json:
            print(f"✓ Confirmed trip JSON: {confirmed_trip_json}")
        else:
            print(f"✗ Confirmed trip JSON: MISSING or EMPTY")
        print()
else:
    print("No entries found in the collection.")

Total entries in 'historical_in_output': 3

=== Entry 1 ===
Trip digest: 2026-07-25: Trip to Amsterdam (Gent ? -> Amsterdam ?)
Session ID: c1368a01
Metadata fields present: ['time_leave', 'timestamp', 'trip_title', 'clarifications', 'trip_date', 'confirmed_trip_json', 'metadata_version', 'original_prompt', 'session_id', 'time_arrival', 'destination', 'origin', 'duration_minutes']
✓ Original prompt: A trip from Gent to Amsterdam, on 25/7, leaving at 8am in gent and returning from Amsterdam at 8pm
✗ Clarifications: NONE (direct confirmation without questions)
✓ Confirmed trip JSON: {"action": "add_trip", "title": "Trip to Amsterdam", "date": "2026-07-25", "from": "Gent", "to": "Amsterdam", "time_leave": "08:00", "time_arrival": "10:25"}

=== Entry 2 ===
Trip digest: 2026-07-25: Trip to Amsterdam - return (Amsterdam ? -> Gent ?)
Session ID: c1368a01
Metadata fields present: ['trip_date', 'confirmed_trip_json', 'metadata_version', 'time_arrival', 'original_prompt', 'trip_title', 'timestamp

### runs code

In [10]:
# SSE-based MCP server connection manager
_mcp_server_process: subprocess.Popen | None = None
_mcp_session: ClientSession | None = None
_mcp_context_manager: Any = None
_mcp_errlog = io.StringIO()


async def connect_to_mcp_server(server_script: str = "mcp/server.py", server_url: str = "http://127.0.0.1:8000/sse") -> ClientSession | None:
    """Start an MCP server subprocess (SSE transport) and connect via HTTP."""
    global _mcp_server_process, _mcp_session, _mcp_context_manager

    if _mcp_session:
        return _mcp_session

    try:
        # Start the MCP server subprocess if not already running
        if _mcp_server_process is None or _mcp_server_process.poll() is not None:
            if isinstance(server_script, str) and server_script.endswith(".py") and os.path.exists(server_script):
                args_list = [server_script]
            else:
                module_name = server_script.replace("/", ".") if isinstance(server_script, str) else str(server_script)
                args_list = ["-m", module_name]

            print(f"Starting MCP server subprocess: {sys.executable} {' '.join(args_list)}")
            _mcp_server_process = subprocess.Popen(
                [sys.executable] + args_list,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
            )
            print(f"MCP server subprocess started (PID={_mcp_server_process.pid})")
            
            # Wait for server to start (give it a few seconds)
            await asyncio.sleep(2)
            
            # Check if process is still running
            if _mcp_server_process.poll() is not None:
                stdout, stderr = _mcp_server_process.communicate()
                print(f"MCP server subprocess exited immediately!")
                print(f"STDOUT: {stdout}")
                print(f"STDERR: {stderr}")
                return None

        # Connect via SSE client
        print(f"Connecting to MCP server via SSE at {server_url}")
        _mcp_context_manager = sse_client(server_url)
        read_stream, write_stream = await _mcp_context_manager.__aenter__()

        session = ClientSession(read_stream, write_stream)
        await session.__aenter__()
        _mcp_session = session

        try:
            await asyncio.wait_for(session.initialize(), timeout=10)
        except asyncio.TimeoutError:
            print("Timeout while initializing MCP session (10s). Aborting connection.")
            try:
                await _mcp_session.__aexit__(None, None, None)
            except Exception:
                pass
            _mcp_session = None
            try:
                await _mcp_context_manager.__aexit__(None, None, None)
            except Exception:
                pass
            _mcp_context_manager = None
            return None

        print(f"Connected to MCP server at {server_url}")
        return session
    except Exception as exc:
        print(f"Failed to connect to MCP server: {exc}")
        import traceback
        traceback.print_exc()
        return None


async def _close_mcp_connection() -> None:
    global _mcp_server_process, _mcp_session, _mcp_context_manager
    if _mcp_session:
        try:
            await _mcp_session.__aexit__(None, None, None)
        except Exception:
            pass
        _mcp_session = None
    if _mcp_context_manager:
        try:
            await _mcp_context_manager.__aexit__(None, None, None)
        except Exception:
            pass
        _mcp_context_manager = None
    if _mcp_server_process and _mcp_server_process.poll() is None:
        try:
            _mcp_server_process.terminate()
            _mcp_server_process.wait(timeout=5)
        except Exception:
            try:
                _mcp_server_process.kill()
            except Exception:
                pass
        _mcp_server_process = None

In [16]:
# Final entry point for the notebook
await main() if asyncio.iscoroutinefunction(main) else main()

Connected to MCP server. Available tools: ['fill_trip_arrival_times', 'fill_trip_distances']
Interactive trip pipeline started. Type 'quit' to stop.

--- Raw prompt sent to model ---
You are an agenda, time-scheduler, and EV charging assistant.

Your capabilities
- Parse natural language descriptions of trips (one-way or round trips)
- Ask clarifying questions when information is missing or ambiguous
- Return structured trip data in JSON format
- Call tools to enrich trip data with distances and arrival times

Trip logic
- A trip can be one-way or part of a round trip.
- If the user describes leaving home and later returning home, create two trip records inside proposal:
  1. Outbound trip: home -> destination
  2. Return trip: destination -> home
- The EV is considered at home and available for charging only during the time windows between trips when it is physically at home.
- Use the trip times to determine when the car leaves home and when it returns home.

Available tools (use whe

In [11]:
def main() -> None:
    """Main entry point with safer Jupyter cleanup behavior."""
    try:
        asyncio.run(rag_pipeline_with_mcp())
    except RuntimeError as e:
        if "asyncio.run() cannot be called from a running event loop" in str(e):
            loop = asyncio.get_event_loop()
            loop.run_until_complete(rag_pipeline_with_mcp())
        else:
            raise
    finally:
        global _mcp_context_manager, _mcp_session, _mcp_errlog
        try:
            loop = asyncio.get_event_loop()
            if loop.is_running():
                print("Event loop is running; to close MCP connections, run: await _close_mcp_connection() in this notebook")
            else:
                loop.run_until_complete(_close_mcp_connection())
        except Exception as exc:
            print(f"Error closing MCP context: {exc}")
        try:
            _mcp_errlog.close()
        except Exception:
            pass


todo: 
* kijken of stuk van advanced prompt van charging calendar ui mbt tot relative dates hier kan toegevoegd worden
* kijken of we die manuele parsing kunnen weglaten
* kijken of we die restrictions kunnen weglaten
* kijken voor UI.

In [15]:
# Enhanced MCP smoke test
session = await connect_to_mcp_server("mcp/server.py")
print("MCP connected:", bool(session))

if session:
    tools = await list_mcp_tools(session)
    print(f"Available tools: {[t.get('name') for t in (tools or [])]}")
    
    # Test a tool call
    test_proposals = {
        "1": {"from": "Gent", "to": "Antwerp"}
    }
    result = await call_mcp_tool(session, "fill_trip_distances", {"proposals": test_proposals})
    print(f"Test tool call result: {result}")
else:
    print("Failed to connect to MCP server")

Connecting to MCP server via SSE at http://127.0.0.1:8000/sse
Connected to MCP server at http://127.0.0.1:8000/sse
MCP connected: True
Available tools: ['fill_trip_arrival_times', 'fill_trip_distances']
Test tool call result: {'proposals': {'1': {'from': 'Gent', 'to': 'Antwerp', 'distance_km': 78.7}}, 'distance_errors': []}


In [22]:
# clear_chromadb()

Deleted collection: charging_plans
Deleted collection: ev_vehicles
Deleted collection: historical_in_output
✓ ChromaDB cleared successfully (3 collections deleted)
